In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path("../rqs/random_selection")

REPOS = [
    "facebook_react",
    "bitcoin_bitcoin",
    "opencv_opencv",
    "tensorflow_tensorflow",
    "microsoft_vscode",
]

MODELS = [
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct",
]

F1_COL = "Macro_F1"


# ============================================================
# LOAD DATA
# ============================================================

dfs = []

for repo in REPOS:
    for model in MODELS:
        provider, model_name = model.split("/", 1)

        csv_path = (
            BASE_DIR
            / repo
            / "models"
            / provider
            / model_name
            / "logs"
            / "fewshot_results_macro_f1.csv"
        )

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)
        df["Repo"] = repo
        df["Model"] = model
        dfs.append(df)

data = pd.concat(dfs, ignore_index=True)


# ============================================================
# STEP 1: MEAN MACRO-F1 PER (Repo, Model, k)
# ============================================================

mean_df = (
    data
    .groupby(["Repo", "Model", "FewShot_Count"], as_index=False)
    .agg(mean_f1=(F1_COL, "mean"))
)


# ============================================================
# STEP 2: CROSS-k SUMMARY PER (Repo, Model)
# ============================================================

summary_df = (
    mean_df
    .groupby(["Repo", "Model"], as_index=False)
    .agg(
        min_f1=("mean_f1", "min"),
        max_f1=("mean_f1", "max"),
    )
)

summary_df["delta_f1"] = summary_df["max_f1"] - summary_df["min_f1"]

summary_df["relative_improvement"] = (
    summary_df["delta_f1"] / summary_df["min_f1"]
) * 100


# Round values
summary_df = summary_df.round({
    "min_f1": 2,
    "max_f1": 2,
    "delta_f1": 2,
    "relative_improvement": 1,
})


# Ensure ordering
summary_df["Repo"] = pd.Categorical(summary_df["Repo"], categories=REPOS, ordered=True)
summary_df["Model"] = pd.Categorical(summary_df["Model"], categories=MODELS, ordered=True)

summary_df = summary_df.sort_values(["Repo", "Model"])


# Reorder columns
summary_df = summary_df[
    [
        "Repo",
        "Model",
        "min_f1",
        "max_f1",
        "delta_f1",
        "relative_improvement",
    ]
]
summary_df

,Repo,Model,min_f1,max_f1,delta_f1,relative_improvement
5,facebook_react,google/gemma-3-12b-it,0.67,0.75,0.08,12.5
6,facebook_react,meta-llama/Llama-3.1-8B-Instruct,0.38,0.47,0.09,22.5
7,facebook_react,mistralai/Mistral-7B-Instruct-v0.3,0.41,0.59,0.18,42.3
4,facebook_react,Qwen/Qwen2.5-7B-Instruct,0.62,0.71,0.08,13.5
1,bitcoin_bitcoin,google/gemma-3-12b-it,0.74,0.77,0.03,3.7
2,bitcoin_bitcoin,meta-llama/Llama-3.1-8B-Instruct,0.49,0.60,0.11,22.3
3,bitcoin_bitcoin,mistralai/Mistral-7B-Instruct-v0.3,0.55,0.69,0.14,24.8
0,bitcoin_bitcoin,Qwen/Qwen2.5-7B-Instruct,0.74,0.80,0.06,7.9
13,opencv_opencv,google/gemma-3-12b-it,0.58,0.71,0.12,21.3
14,opencv_opencv,meta-llama/Llama-3.1-8B-Instruct,0.23,0.30,0.07,32.3


In [4]:
# ============================================================
# GENERATE LATEX TABLE
# ============================================================

def format_f1(x):
    return f"{x:.2f}"

def format_percent(x):
    return f"{x:.1f}\\%"


repo_display = {
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "facebook_react": "React",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode": "VsCode",
}

model_display = {
    "meta-llama/Llama-3.1-8B-Instruct": "Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3": "Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct": "Qwen2.5-7B-Instruct",
    "google/gemma-3-12b-it": "Gemma-3-12b-it",
}


print("\\begin{table*}[t]")
print("    \\centering")
print("    \\caption{Macro-F1 range across few-shot counts per repository and model.}")
print("    \\label{tab:f1-range-stratified-selection-cross-k}")
print("    \\resizebox{\\textwidth}{!}{")
print("        \\begin{tabular}{llrrrrrr}")
print("            \\toprule")
print("            \\textbf{Repository} & \\textbf{Model} & "
      "\\textbf{Min F1} & \\textbf{Max F1} & "
      "\\textbf{Mean F1} & \\textbf{Median F1} & "
      "\\textbf{$\\Delta$F1} & \\textbf{Rel. Impr.} \\\\")
print("            \\midrule")


repos = summary_df["Repo"].unique()

for i, repo in enumerate(repos):

    repo_df = summary_df[summary_df["Repo"] == repo]
    repo_name = repo_display.get(repo, repo)
    n_models = len(repo_df)

    for j, (_, row) in enumerate(repo_df.iterrows()):

        model_name = model_display.get(row["Model"], row["Model"])

        min_f1 = format_f1(row["min_f1"])
        max_f1 = format_f1(row["max_f1"])
        mean_f1 = format_f1(row["mean_over_k"])
        median_f1 = format_f1(row["median_over_k"])
        delta = format_f1(row["delta_f1"])
        rel = format_percent(row["relative_improvement"])

        if j == 0:
            print(f"            \\multirow{{{n_models}}}{{*}}{{{repo_name}}} "
                  f"& {model_name} & {min_f1} & {max_f1} & "
                  f"{mean_f1} & {median_f1} & {delta} & {rel} \\\\")
        else:
            print(f"             & {model_name} & {min_f1} & {max_f1} & "
                  f"{mean_f1} & {median_f1} & {delta} & {rel} \\\\")

    if i < len(repos) - 1:
        print("            \\midrule")


print("            \\bottomrule")
print("        \\end{tabular}")
print("    }")
print("\\end{table*}")

\begin{table*}[t]
    \centering
    \caption{Macro-F1 range across few-shot counts per repository and model.}
    \label{tab:f1-range-stratified-selection-cross-k}
    \resizebox{\textwidth}{!}{
        \begin{tabular}{llrrrrrr}
            \toprule
            \textbf{Repository} & \textbf{Model} & \textbf{Min F1} & \textbf{Max F1} & \textbf{Mean F1} & \textbf{Median F1} & \textbf{$\Delta$F1} & \textbf{Rel. Impr.} \\
            \midrule
            \multirow{4}{*}{React} & Gemma-3-12b-it & 0.67 & 0.75 & 0.71 & 0.72 & 0.08 & 12.5\% \\
             & Llama-3.1-8B-Instruct & 0.38 & 0.47 & 0.42 & 0.42 & 0.09 & 22.5\% \\
             & Mistral-7B-Instruct-v0.3 & 0.41 & 0.59 & 0.54 & 0.56 & 0.18 & 42.3\% \\
             & Qwen2.5-7B-Instruct & 0.62 & 0.71 & 0.67 & 0.66 & 0.08 & 13.5\% \\
            \midrule
            \multirow{4}{*}{Bitcoin} & Gemma-3-12b-it & 0.74 & 0.77 & 0.75 & 0.75 & 0.03 & 3.7\% \\
             & Llama-3.1-8B-Instruct & 0.49 & 0.60 & 0.55 & 0.54 & 0.11 & 22.3\% \\

In [2]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

ZERO_SHOT_BASE_DIR = Path("../rqs/zero_shot")

REPOS = [
    "facebook_react",
    "bitcoin_bitcoin",
    "opencv_opencv",
    "tensorflow_tensorflow",
    "microsoft_vscode",
]

MODELS = [
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct",
]

REPO_DISPLAY = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode": "VsCode",
}

MODEL_DISPLAY = {
    "google/gemma-3-12b-it": "Gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B-Instruct": "Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3": "Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct": "Qwen2.5-7B-Instruct",
}

# ============================================================
# LOAD RESULTS
# ============================================================

rows = []

for repo in REPOS:
    for model in MODELS:

        provider, model_name = model.split("/", 1)

        csv_path = (
            ZERO_SHOT_BASE_DIR
            / repo
            / provider
            / model_name
            / "logs"
            / "zero_shot_results_macro_f1.csv"
        )

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)

        macro_f1 = df.iloc[0]["Macro_F1"]

        rows.append({
            "repo": repo,
            "model": model,
            "Macro_F1": macro_f1
        })

results = pd.DataFrame(rows)

# Round F1 values
results["Macro_F1"] = results["Macro_F1"].round(2)

# Ensure ordering
results["repo"] = pd.Categorical(results["repo"], categories=REPOS, ordered=True)
results["model"] = pd.Categorical(results["model"], categories=MODELS, ordered=True)

results = results.sort_values(["repo", "model"])


# ============================================================
# GENERATE LATEX TABLE
# ============================================================

lines = []

lines.append("\\begin{table}[t]")
lines.append("\\centering")
lines.append("\\caption{Zero-shot Macro-F1 performance per repository and model.}")
lines.append("\\label{tab:zero-shot-macro-f1}")
lines.append("\\begin{tabular}{llr}")
lines.append("\\toprule")
lines.append("\\textbf{Repository} & \\textbf{Model} & \\textbf{Macro-F1} \\\\")
lines.append("\\midrule")

for repo in REPOS:

    repo_df = results[results.repo == repo]

    if repo_df.empty:
        continue

    first = True

    for _, row in repo_df.iterrows():

        repo_name = REPO_DISPLAY[row.repo]
        model_name = MODEL_DISPLAY[row.model]
        f1 = row.Macro_F1

        if first:
            lines.append(
                f"\\multirow{{{len(repo_df)}}}{{*}}{{{repo_name}}} & {model_name} & {f1:.2f} \\\\"
            )
            first = False
        else:
            lines.append(
                f"& {model_name} & {f1:.2f} \\\\"
            )

    lines.append("\\midrule")

lines.pop()  # remove last midrule

lines.append("\\bottomrule")
lines.append("\\end{tabular}")
lines.append("\\end{table}")

latex_code = "\n".join(lines)

print(latex_code)

\begin{table}[t]
\centering
\caption{Zero-shot Macro-F1 performance per repository and model.}
\label{tab:zero-shot-macro-f1}
\begin{tabular}{llr}
\toprule
\textbf{Repository} & \textbf{Model} & \textbf{Macro-F1} \\
\midrule
\multirow{4}{*}{React} & Gemma-3-12b-it & 0.64 \\
& Llama-3.1-8B-Instruct & 0.27 \\
& Mistral-7B-Instruct-v0.3 & 0.43 \\
& Qwen2.5-7B-Instruct & 0.64 \\
\midrule
\multirow{4}{*}{Bitcoin} & Gemma-3-12b-it & 0.71 \\
& Llama-3.1-8B-Instruct & 0.41 \\
& Mistral-7B-Instruct-v0.3 & 0.62 \\
& Qwen2.5-7B-Instruct & 0.77 \\
\midrule
\multirow{4}{*}{OpenCV} & Gemma-3-12b-it & 0.55 \\
& Llama-3.1-8B-Instruct & 0.27 \\
& Mistral-7B-Instruct-v0.3 & 0.32 \\
& Qwen2.5-7B-Instruct & 0.61 \\
\midrule
\multirow{4}{*}{Tensorflow} & Gemma-3-12b-it & 0.66 \\
& Llama-3.1-8B-Instruct & 0.34 \\
& Mistral-7B-Instruct-v0.3 & 0.44 \\
& Qwen2.5-7B-Instruct & 0.64 \\
\midrule
\multirow{4}{*}{VsCode} & Gemma-3-12b-it & 0.40 \\
& Llama-3.1-8B-Instruct & 0.16 \\
& Mistral-7B-Instruct-v0.3 & 0.32 

In [11]:
# ============================================================
# MERGE ZERO-SHOT + FEW-SHOT
# ============================================================

combined = pd.merge(
    summary_df,
    results,
    left_on=["Repo", "Model"],
    right_on=["repo", "model"],
    how="left"
)

# Clean columns
combined = combined.rename(columns={
    "Macro_F1": "zero_shot_f1"
})

combined = combined[
    [
        "Repo",
        "Model",
        "zero_shot_f1",
        "min_f1",
        "max_f1",
        "delta_f1",
        "relative_improvement",
    ]
]

# ============================================================
# FORMATTERS
# ============================================================

def format_f1(x):
    if pd.isna(x):
        return "-"
    return f"{x:.2f}"

def format_percent(x):
    if pd.isna(x):
        return "-"
    return f"{x:.1f}\\%"


# ============================================================
# GENERATE LATEX TABLE
# ============================================================

print("\\begin{table*}[t]")
print("    \\centering")
print("    \\caption{Zero-shot vs Few-shot Macro-F1 performance across repositories and models.}")
print("    \\label{tab:zero-vs-few-shot}")
print("    \\resizebox{\\textwidth}{!}{")
print("        \\begin{tabular}{llr|rrrrrr}")
print("            \\toprule")

# Header row 1
print("            & & \\multicolumn{1}{c}{\\textbf{Zero-shot}} "
      "& \\multicolumn{6}{c}{\\textbf{Few-shot}} \\\\")

# Header row 2
print("            \\textbf{Repository} & \\textbf{Model} & \\textbf{Macro-F1} "
      "& \\textbf{Min} & \\textbf{Max} & \\textbf{Mean} & "
      "\\textbf{Median} & \\textbf{$\\Delta$} & \\textbf{Rel. (\\%)} \\\\")

print("            \\midrule")


repos = combined["Repo"].unique()

for i, repo in enumerate(repos):

    repo_df = combined[combined["Repo"] == repo]
    repo_name = repo_display.get(repo, repo)
    n_models = len(repo_df)

    for j, (_, row) in enumerate(repo_df.iterrows()):

        model_name = model_display.get(row["Model"], row["Model"])

        zero = format_f1(row["zero_shot_f1"])
        min_f1 = format_f1(row["min_f1"])
        max_f1 = format_f1(row["max_f1"])
        mean_f1 = format_f1(row["mean_over_k"])
        median_f1 = format_f1(row["median_over_k"])
        delta = format_f1(row["delta_f1"])
        rel = format_percent(row["relative_improvement"])

        if j == 0:
            print(
                f"            \\multirow{{{n_models}}}{{*}}{{{repo_name}}} "
                f"& {model_name} & {zero} "
                f"& {min_f1} & {max_f1} & {mean_f1} & {median_f1} & {delta} & {rel} \\\\"
            )
        else:
            print(
                f"             & {model_name} & {zero} "
                f"& {min_f1} & {max_f1} & {mean_f1} & {median_f1} & {delta} & {rel} \\\\"
            )

    if i < len(repos) - 1:
        print("            \\midrule")

print("            \\bottomrule")
print("        \\end{tabular}")
print("    }")
print("\\end{table*}")

\begin{table*}[t]
    \centering
    \caption{Zero-shot vs Few-shot Macro-F1 performance across repositories and models.}
    \label{tab:zero-vs-few-shot}
    \resizebox{\textwidth}{!}{
        \begin{tabular}{llr|rrrrrr}
            \toprule
            & & \multicolumn{1}{c}{\textbf{Zero-shot}} & \multicolumn{6}{c}{\textbf{Few-shot}} \\
            \textbf{Repository} & \textbf{Model} & \textbf{Macro-F1} & \textbf{Min} & \textbf{Max} & \textbf{Mean} & \textbf{Median} & \textbf{$\Delta$} & \textbf{Rel. (\%)} \\
            \midrule
            \multirow{4}{*}{React} & Gemma-3-12b-it & 0.64 & 0.67 & 0.75 & 0.71 & 0.72 & 0.08 & 12.5\% \\
             & Llama-3.1-8B-Instruct & 0.27 & 0.38 & 0.47 & 0.42 & 0.42 & 0.09 & 22.5\% \\
             & Mistral-7B-Instruct-v0.3 & 0.43 & 0.41 & 0.59 & 0.54 & 0.56 & 0.18 & 42.3\% \\
             & Qwen2.5-7B-Instruct & 0.64 & 0.62 & 0.71 & 0.67 & 0.66 & 0.08 & 13.5\% \\
            \midrule
            \multirow{4}{*}{Bitcoin} & Gemma-3-12b-it & 0.71

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

FEWSHOT_BASE_DIR = Path("../rqs/random_selection")
ZERO_SHOT_BASE_DIR = Path("../rqs/zero_shot")

REPOS = [
    "facebook_react",
    "bitcoin_bitcoin",
    "opencv_opencv",
    "tensorflow_tensorflow",
    "microsoft_vscode",
]

MODELS = [
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct",
]

F1_COL = "Macro_F1"

REPO_DISPLAY = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "TensorFlow",
    "microsoft_vscode": "VSCode",
}

MODEL_DISPLAY = {
    "google/gemma-3-12b-it": "Gemma-3-12B-it",
    "meta-llama/Llama-3.1-8B-Instruct": "Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3": "Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct": "Qwen2.5-7B-Instruct",
}

# ============================================================
# LOAD FEW-SHOT RESULTS
# ============================================================

fewshot_dfs = []

for repo in REPOS:
    for model in MODELS:
        provider, model_name = model.split("/", 1)

        csv_path = (
            FEWSHOT_BASE_DIR
            / repo
            / "models"
            / provider
            / model_name
            / "logs"
            / "fewshot_results_macro_f1.csv"
        )

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)
        df[F1_COL] = pd.to_numeric(df[F1_COL], errors="coerce")
        df = df.dropna(subset=[F1_COL])

        df["Repo"] = repo
        df["Model"] = model
        fewshot_dfs.append(df)

fewshot_data = pd.concat(fewshot_dfs, ignore_index=True)

# ============================================================
# FEW-SHOT SUMMARY
# ============================================================

# Mean Macro-F1 for each (Repo, Model, k)
mean_df = (
    fewshot_data
    .groupby(["Repo", "Model", "FewShot_Count"], as_index=False)
    .agg(mean_f1=(F1_COL, "mean"))
)

# Summary across k for each (Repo, Model)
summary_df = (
    mean_df
    .groupby(["Repo", "Model"], as_index=False)
    .agg(
        min_f1=("mean_f1", "min"),
        max_f1=("mean_f1", "max"),
        mean_over_k=("mean_f1", "mean"),
        median_over_k=("mean_f1", "median"),
    )
)

summary_df["delta_f1"] = summary_df["max_f1"] - summary_df["min_f1"]

summary_df["relative_variation_pct"] = (
    summary_df["delta_f1"] / summary_df["min_f1"]
) * 100

# ============================================================
# LOAD ZERO-SHOT RESULTS
# ============================================================

zero_rows = []

for repo in REPOS:
    for model in MODELS:
        provider, model_name = model.split("/", 1)

        csv_path = (
            ZERO_SHOT_BASE_DIR
            / repo
            / provider
            / model_name
            / "logs"
            / "zero_shot_results_macro_f1.csv"
        )

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)
        df[F1_COL] = pd.to_numeric(df[F1_COL], errors="coerce")
        df = df.dropna(subset=[F1_COL])

        if df.empty:
            continue

        zero_f1 = df.iloc[0][F1_COL]

        zero_rows.append({
            "Repo": repo,
            "Model": model,
            "zero_shot_f1": zero_f1,
        })

zero_df = pd.DataFrame(zero_rows)

# ============================================================
# MERGE ZERO-SHOT + FEW-SHOT
# ============================================================

combined = pd.merge(
    summary_df,
    zero_df,
    on=["Repo", "Model"],
    how="left"
)

combined["improvement_over_zero_f1"] = (
    combined["max_f1"] - combined["zero_shot_f1"]
)

combined["relative_improvement_over_zero_pct"] = (
    combined["improvement_over_zero_f1"] / combined["zero_shot_f1"]
) * 100

# ============================================================
# ORDERING + ROUNDING
# ============================================================

combined["Repo"] = pd.Categorical(combined["Repo"], categories=REPOS, ordered=True)
combined["Model"] = pd.Categorical(combined["Model"], categories=MODELS, ordered=True)

combined = combined.sort_values(["Repo", "Model"]).reset_index(drop=True)

combined = combined.round({
    "zero_shot_f1": 2,
    "min_f1": 2,
    "max_f1": 2,
    "mean_over_k": 2,
    "median_over_k": 2,
    "delta_f1": 2,
    "relative_variation_pct": 1,
    "improvement_over_zero_f1": 2,
    "relative_improvement_over_zero_pct": 1,
})

# ============================================================
# OPTIONAL: VIEW DATAFRAME
# ============================================================

print("\n===== Combined Summary =====\n")
print(combined)

# ============================================================
# FORMATTERS
# ============================================================

def format_f1(x):
    if pd.isna(x):
        return "-"
    return f"{x:.2f}"

def format_percent(x):
    if pd.isna(x):
        return "-"
    return f"{x:.1f}\\%"

# ============================================================
# GENERATE LATEX TABLE
# ============================================================

print("\\begin{table*}[t]")
print("    \\centering")
print("    \\caption{Zero-shot and few-shot Macro-F1 performance across repositories and models.}")
print("    \\label{tab:zero-vs-few-shot}")
print("    \\resizebox{\\textwidth}{!}{")
print("        \\begin{tabular}{llr|rrrrrrr}")
print("            \\toprule")

print(
    "            & & \\multicolumn{1}{c}{\\textbf{Zero-shot}} "
    "& \\multicolumn{7}{c}{\\textbf{Few-shot}} \\\\"
)

print(
    "            \\textbf{Repository} & \\textbf{Model} & \\textbf{Macro-F1} "
    "& \\textbf{Min} & \\textbf{Max} & \\textbf{Mean} & \\textbf{Median} "
    "& \\textbf{$\\Delta$} & \\textbf{Rel. Var. (\\%)} & \\textbf{Rel. Impr. vs Zero (\\%)} \\\\"
)

print("            \\midrule")

repos = combined["Repo"].dropna().unique()

for i, repo in enumerate(repos):
    repo_df = combined[combined["Repo"] == repo]
    repo_name = REPO_DISPLAY.get(str(repo), str(repo))
    n_models = len(repo_df)

    for j, (_, row) in enumerate(repo_df.iterrows()):
        model_name = MODEL_DISPLAY.get(str(row["Model"]), str(row["Model"]))

        zero = format_f1(row["zero_shot_f1"])
        min_f1 = format_f1(row["min_f1"])
        max_f1 = format_f1(row["max_f1"])
        mean_f1 = format_f1(row["mean_over_k"])
        median_f1 = format_f1(row["median_over_k"])
        delta = format_f1(row["delta_f1"])
        rel_var = format_percent(row["relative_variation_pct"])
        rel_impr_zero = format_percent(row["relative_improvement_over_zero_pct"])

        if j == 0:
            print(
                f"            \\multirow{{{n_models}}}{{*}}{{{repo_name}}} "
                f"& {model_name} & {zero} "
                f"& {min_f1} & {max_f1} & {mean_f1} & {median_f1} "
                f"& {delta} & {rel_var} & {rel_impr_zero} \\\\"
            )
        else:
            print(
                f"            & {model_name} & {zero} "
                f"& {min_f1} & {max_f1} & {mean_f1} & {median_f1} "
                f"& {delta} & {rel_var} & {rel_impr_zero} \\\\"
            )

    if i < len(repos) - 1:
        print("            \\midrule")

print("            \\bottomrule")
print("        \\end{tabular}")
print("    }")
print("\\end{table*}")


===== Combined Summary =====

                     Repo                               Model  min_f1  max_f1  \
0          facebook_react               google/gemma-3-12b-it    0.67    0.75   
1          facebook_react    meta-llama/Llama-3.1-8B-Instruct    0.38    0.47   
2          facebook_react  mistralai/Mistral-7B-Instruct-v0.3    0.41    0.59   
3          facebook_react            Qwen/Qwen2.5-7B-Instruct    0.62    0.71   
4         bitcoin_bitcoin               google/gemma-3-12b-it    0.74    0.77   
5         bitcoin_bitcoin    meta-llama/Llama-3.1-8B-Instruct    0.49    0.60   
6         bitcoin_bitcoin  mistralai/Mistral-7B-Instruct-v0.3    0.55    0.69   
7         bitcoin_bitcoin            Qwen/Qwen2.5-7B-Instruct    0.74    0.80   
8           opencv_opencv               google/gemma-3-12b-it    0.58    0.71   
9           opencv_opencv    meta-llama/Llama-3.1-8B-Instruct    0.23    0.30   
10          opencv_opencv  mistralai/Mistral-7B-Instruct-v0.3    0.35    0.46 

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

FEWSHOT_BASE_DIR = Path("../rqs/random_selection")
ZERO_SHOT_BASE_DIR = Path("../rqs/zero_shot")

REPOS = [
    "facebook_react",
    "bitcoin_bitcoin",
    "opencv_opencv",
    "tensorflow_tensorflow",
    "microsoft_vscode",
]

MODELS = [
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct",
]

F1_COL = "Macro_F1"

REPO_DISPLAY = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "TensorFlow",
    "microsoft_vscode": "VSCode",
}

MODEL_DISPLAY = {
    "google/gemma-3-12b-it": "Gemma-3-12B-it",
    "meta-llama/Llama-3.1-8B-Instruct": "Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3": "Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct": "Qwen2.5-7B-Instruct",
}

# ============================================================
# LOAD FEW-SHOT RESULTS
# ============================================================

fewshot_dfs = []

for repo in REPOS:
    for model in MODELS:
        provider, model_name = model.split("/", 1)

        csv_path = (
            FEWSHOT_BASE_DIR
            / repo
            / "models"
            / provider
            / model_name
            / "logs"
            / "fewshot_results_macro_f1.csv"
        )

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)
        df[F1_COL] = pd.to_numeric(df[F1_COL], errors="coerce")
        df = df.dropna(subset=[F1_COL])

        df["Repo"] = repo
        df["Model"] = model
        fewshot_dfs.append(df)

fewshot_data = pd.concat(fewshot_dfs, ignore_index=True)

# ============================================================
# FEW-SHOT SUMMARY
# ============================================================

# Mean Macro-F1 for each (Repo, Model, k)
mean_df = (
    fewshot_data
    .groupby(["Repo", "Model", "FewShot_Count"], as_index=False)
    .agg(mean_f1=(F1_COL, "mean"))
)

# Summary across k for each (Repo, Model)
summary_df = (
    mean_df
    .groupby(["Repo", "Model"], as_index=False)
    .agg(
        min_f1=("mean_f1", "min"),
        max_f1=("mean_f1", "max"),
    )
)

summary_df["delta_f1"] = summary_df["max_f1"] - summary_df["min_f1"]

summary_df["relative_variation_pct"] = (
    summary_df["delta_f1"] / summary_df["min_f1"]
) * 100

# ============================================================
# LOAD ZERO-SHOT RESULTS
# ============================================================

zero_rows = []

for repo in REPOS:
    for model in MODELS:
        provider, model_name = model.split("/", 1)

        csv_path = (
            ZERO_SHOT_BASE_DIR
            / repo
            / provider
            / model_name
            / "logs"
            / "zero_shot_results_macro_f1.csv"
        )

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)
        df[F1_COL] = pd.to_numeric(df[F1_COL], errors="coerce")
        df = df.dropna(subset=[F1_COL])

        if df.empty:
            continue

        zero_f1 = df.iloc[0][F1_COL]

        zero_rows.append({
            "Repo": repo,
            "Model": model,
            "zero_shot_f1": zero_f1,
        })

zero_df = pd.DataFrame(zero_rows)

# ============================================================
# MERGE ZERO-SHOT + FEW-SHOT
# ============================================================

combined = pd.merge(
    summary_df,
    zero_df,
    on=["Repo", "Model"],
    how="left"
)

combined["improvement_over_zero_f1"] = (
    combined["max_f1"] - combined["zero_shot_f1"]
)

combined["relative_improvement_over_zero_pct"] = (
    combined["improvement_over_zero_f1"] / combined["zero_shot_f1"]
) * 100

# ============================================================
# ORDERING + ROUNDING
# ============================================================

combined["Repo"] = pd.Categorical(combined["Repo"], categories=REPOS, ordered=True)
combined["Model"] = pd.Categorical(combined["Model"], categories=MODELS, ordered=True)

combined = combined.sort_values(["Repo", "Model"]).reset_index(drop=True)

combined = combined.round({
    "zero_shot_f1": 2,
    "min_f1": 2,
    "max_f1": 2,
    "delta_f1": 2,
    "relative_variation_pct": 1,
    "improvement_over_zero_f1": 2,
    "relative_improvement_over_zero_pct": 1,
})

# ============================================================
# FORMATTERS
# ============================================================

def format_f1(x):
    if pd.isna(x):
        return "-"
    return f"{x:.2f}"

def format_percent(x):
    if pd.isna(x):
        return "-"
    return f"{x:.1f}\\%"

# ============================================================
# GENERATE LATEX TABLE
# ============================================================

print("\\begin{table*}[t]")
print("    \\centering")
print("    \\caption{Zero-shot and few-shot Macro-F1 performance across repositories and models.}")
print("    \\label{tab:zero-vs-few-shot}")
print("    \\resizebox{\\textwidth}{!}{")
print("        \\begin{tabular}{llr|rrrrr}")
print("            \\toprule")

print(
    "            & & \\multicolumn{1}{c}{\\textbf{Zero-shot}} "
    "& \\multicolumn{5}{c}{\\textbf{Few-shot}} \\\\"
)

print(
    "            \\textbf{Repository} & \\textbf{Model} & \\textbf{Macro-F1} "
    "& \\textbf{Min} & \\textbf{Max} & \\textbf{$\\Delta$} "
    "& \\textbf{Rel. Var. (\\%)} & \\textbf{Rel. Impr. vs Zero (\\%)} \\\\"
)

print("            \\midrule")

repos = combined["Repo"].dropna().unique()

for i, repo in enumerate(repos):
    repo_df = combined[combined["Repo"] == repo]
    repo_name = REPO_DISPLAY.get(str(repo), str(repo))
    n_models = len(repo_df)

    for j, (_, row) in enumerate(repo_df.iterrows()):
        model_name = MODEL_DISPLAY.get(str(row["Model"]), str(row["Model"]))

        zero = format_f1(row["zero_shot_f1"])
        min_f1 = format_f1(row["min_f1"])
        max_f1 = format_f1(row["max_f1"])
        delta = format_f1(row["delta_f1"])
        rel_var = format_percent(row["relative_variation_pct"])
        rel_impr_zero = format_percent(row["relative_improvement_over_zero_pct"])

        if j == 0:
            print(
                f"            \\multirow{{{n_models}}}{{*}}{{{repo_name}}} "
                f"& {model_name} & {zero} "
                f"& {min_f1} & {max_f1} & {delta} & {rel_var} & {rel_impr_zero} \\\\"
            )
        else:
            print(
                f"            & {model_name} & {zero} "
                f"& {min_f1} & {max_f1} & {delta} & {rel_var} & {rel_impr_zero} \\\\"
            )

    if i < len(repos) - 1:
        print("            \\midrule")

print("            \\bottomrule")
print("        \\end{tabular}")
print("    }")
print("\\end{table*}")

\begin{table*}[t]
    \centering
    \caption{Zero-shot and few-shot Macro-F1 performance across repositories and models.}
    \label{tab:zero-vs-few-shot}
    \resizebox{\textwidth}{!}{
        \begin{tabular}{llr|rrrrr}
            \toprule
            & & \multicolumn{1}{c}{\textbf{Zero-shot}} & \multicolumn{5}{c}{\textbf{Few-shot}} \\
            \textbf{Repository} & \textbf{Model} & \textbf{Macro-F1} & \textbf{Min} & \textbf{Max} & \textbf{$\Delta$} & \textbf{Rel. Var. (\%)} & \textbf{Rel. Impr. vs Zero (\%)} \\
            \midrule
            \multirow{4}{*}{React} & Gemma-3-12B-it & 0.64 & 0.67 & 0.75 & 0.08 & 12.5\% & 17.9\% \\
            & Llama-3.1-8B-Instruct & 0.27 & 0.38 & 0.47 & 0.09 & 22.5\% & 72.7\% \\
            & Mistral-7B-Instruct-v0.3 & 0.43 & 0.41 & 0.59 & 0.18 & 42.3\% & 37.8\% \\
            & Qwen2.5-7B-Instruct & 0.64 & 0.62 & 0.71 & 0.08 & 13.5\% & 9.8\% \\
            \midrule
            \multirow{4}{*}{Bitcoin} & Gemma-3-12B-it & 0.71 & 0.74 & 0.77 & 0

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

RANDOM_BASE_DIR = Path("../rqs/random_selection")
SIM_BASE_DIR = Path("../rqs/similarity_selection_RAG")

TARGET_K = 8
F1_COL = "Macro_F1"

ALLOWED_MODELS = [
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Qwen2.5-7B-Instruct",
]

REPO_NAME_MAP = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "OpenCV",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode": "VsCode",
}

EMBED_MODEL = "all-mpnet-base-v2"

# ============================================================
# LOAD RANDOM SELECTION (k = 8)
# ============================================================

random_rows = []

for repo_dir in RANDOM_BASE_DIR.iterdir():

    if not repo_dir.is_dir():
        continue

    repo = repo_dir.name
    models_dir = repo_dir / "models"

    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():
        if not provider_dir.is_dir():
            continue

        for model_dir in provider_dir.iterdir():

            model_name = model_dir.name

            if model_name not in ALLOWED_MODELS:
                continue

            csv_path = model_dir / "logs" / "fewshot_results_macro_f1.csv"

            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]

            if not {"FewShot_Count", F1_COL, "Token_Count"}.issubset(df.columns):
                continue

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df[F1_COL] = pd.to_numeric(df[F1_COL], errors="coerce")
            df["Token_Count"] = pd.to_numeric(df["Token_Count"], errors="coerce")

            df = df.dropna(subset=["FewShot_Count", F1_COL])

            df = df[df["FewShot_Count"] == TARGET_K]

            if df.empty:
                continue

            f1_values = df[F1_COL].values

            min_f1 = np.min(f1_values)
            max_f1 = np.max(f1_values)
            mean_f1 = np.mean(f1_values)
            median_f1 = np.median(f1_values)

            delta_f1 = max_f1 - min_f1
            rel_impr = np.inf if min_f1 == 0 else (delta_f1 / min_f1) * 100

            # token count for max F1 row
            max_row = df.loc[df[F1_COL].idxmax()]
            max_token = max_row["Token_Count"]

            random_rows.append({
                "Repository": REPO_NAME_MAP.get(repo, repo),
                "Model": model_name,

                "min_f1": min_f1,
                "max_f1": max_f1,
                "mean_f1": mean_f1,
                "median_f1": median_f1,
                "delta_f1": delta_f1,
                "rel_impr": rel_impr,

                "max_token": max_token
            })

random_df = pd.DataFrame(random_rows)

# ============================================================
# LOAD SIMILARITY SELECTION (k = 8)
# ============================================================

sim_rows = []

for repo_dir in SIM_BASE_DIR.iterdir():

    if not repo_dir.is_dir():
        continue

    repo = repo_dir.name
    models_dir = repo_dir / EMBED_MODEL / "models"

    if not models_dir.exists():
        continue

    for provider_dir in models_dir.iterdir():
        if not provider_dir.is_dir():
            continue

        for model_dir in provider_dir.iterdir():

            model_name = model_dir.name

            if model_name not in ALLOWED_MODELS:
                continue

            csv_path = model_dir / "logs" / "similarity_results_macro_f1.csv"

            if not csv_path.exists():
                continue

            df = pd.read_csv(csv_path)

            if not {"FewShot_Count", "Macro_F1", "Token_Count"}.issubset(df.columns):
                continue

            df["FewShot_Count"] = pd.to_numeric(df["FewShot_Count"], errors="coerce")
            df["Macro_F1"] = pd.to_numeric(df["Macro_F1"], errors="coerce")
            df["Token_Count"] = pd.to_numeric(df["Token_Count"], errors="coerce")

            df = df.dropna(subset=["FewShot_Count", "Macro_F1"])

            df = df[df["FewShot_Count"] == TARGET_K]

            if df.empty:
                continue

            sim_rows.append({
                "Repository": REPO_NAME_MAP.get(repo, repo),
                "Model": model_name,
                "sim_f1": df["Macro_F1"].values[0],
                "sim_token": df["Token_Count"].values[0]
            })

sim_df = pd.DataFrame(sim_rows)

# ============================================================
# MERGE
# ============================================================

combined = pd.merge(
    sim_df,
    random_df,
    on=["Repository", "Model"],
    how="inner"
)

combined = combined.sort_values(["Repository", "Model"])

# ============================================================
# HELPERS
# ============================================================

def fmt_f1(x):
    return r"$\infty$" if x == float("inf") else f"{x:.2f}"

def fmt_int(x):
    return f"{int(x)}"

def fmt_percent(x):
    return r"$\infty$" if x == float("inf") else f"{x:.1f}\\%"

# ============================================================
# LATEX TABLE
# ============================================================

print("\\begin{table*}[t]")
print("    \\centering")
print("    \\caption{Similarity vs Random few-shot selection at $k=8$.}")
print("    \\label{tab:sim-vs-random}")
print("    \\resizebox{\\textwidth}{!}{")
print("        \\begin{tabular}{llrr|rrrrrrr}")
print("            \\toprule")

# header 1
print("            & & \\multicolumn{2}{c}{\\textbf{Similarity}} "
      "& \\multicolumn{7}{c}{\\textbf{Random}} \\\\")

# header 2
print("            \\textbf{Repository} & \\textbf{Model} "
      "& \\textbf{F1} & \\textbf{Tokens} "
      "& \\textbf{Min} & \\textbf{Max} & \\textbf{Mean} & \\textbf{Median} "
      "& \\textbf{$\\Delta$} & \\textbf{Rel. (\\%)} & \\textbf{Max Tokens} \\\\")

print("            \\midrule")

repos = combined["Repository"].unique()

for i, repo in enumerate(repos):

    repo_df = combined[combined["Repository"] == repo]
    n_models = len(repo_df)

    for j, (_, r) in enumerate(repo_df.iterrows()):

        repo_cell = f"\\multirow{{{n_models}}}{{*}}{{{repo}}}" if j == 0 else ""

        print(
            f"            {repo_cell} & {r['Model']} "
            f"& {fmt_f1(r['sim_f1'])} & {fmt_int(r['sim_token'])} "
            f"& {fmt_f1(r['min_f1'])} & {fmt_f1(r['max_f1'])} "
            f"& {fmt_f1(r['mean_f1'])} & {fmt_f1(r['median_f1'])} "
            f"& {fmt_f1(r['delta_f1'])} & {fmt_percent(r['rel_impr'])} "
            f"& {fmt_int(r['max_token'])} \\\\"
        )

    if i < len(repos) - 1:
        print("            \\midrule")

print("            \\bottomrule")
print("        \\end{tabular}")
print("    }")
print("\\end{table*}")

\begin{table*}[t]
    \centering
    \caption{Similarity vs Random few-shot selection at $k=8$.}
    \label{tab:sim-vs-random}
    \resizebox{\textwidth}{!}{
        \begin{tabular}{llrr|rrrrrrr}
            \toprule
            & & \multicolumn{2}{c}{\textbf{Similarity}} & \multicolumn{7}{c}{\textbf{Random}} \\
            \textbf{Repository} & \textbf{Model} & \textbf{F1} & \textbf{Tokens} & \textbf{Min} & \textbf{Max} & \textbf{Mean} & \textbf{Median} & \textbf{$\Delta$} & \textbf{Rel. (\%)} & \textbf{Max Tokens} \\
            \midrule
            \multirow{4}{*}{Bitcoin} & Llama-3.1-8B-Instruct & 0.58 & 2881 & 0.36 & 0.71 & 0.53 & 0.53 & 0.35 & 95.8\% & 2613 \\
             & Mistral-7B-Instruct-v0.3 & 0.68 & 3502 & 0.63 & 0.71 & 0.67 & 0.67 & 0.09 & 13.9\% & 7518 \\
             & Qwen2.5-7B-Instruct & 0.80 & 3122 & 0.69 & 0.81 & 0.77 & 0.77 & 0.12 & 17.0\% & 3426 \\
             & gemma-3-12b-it & 0.79 & 3243 & 0.73 & 0.79 & 0.76 & 0.76 & 0.07 & 9.1\% & 1983 \\
            \midr